In [2]:
import numpy as np
import pandas as pd

from cvxopt import matrix, solvers
from sklearn.model_selection import train_test_split


def generate_wsvcr_data(n=500, censoring_rate=0.3, random_state=42):
    np.random.seed(random_state)

    x1 = np.random.normal(1, 1, n)
    x2 = np.random.normal(2, np.sqrt(2), n)
    x3 = np.random.exponential(1, n)

    x2_safe = np.where(np.abs(x2) < 1e-8, 1e-8, x2)

    lam = 5 * np.exp(
        -x1 * np.log(np.abs(x2_safe)) + np.sin(x3 ** 2)
    )

    k = 3
    U = np.random.uniform(0, 1, n)
    Y = lam * (-np.log1p(-U)) ** (1 / k)

    def generate_censoring(Y, target_rate, tol=0.01, max_iter=50):
        scale = np.median(Y)

        for _ in range(max_iter):
            C = np.random.exponential(scale=scale, size=len(Y))
            delta = (Y <= C).astype(int)
            rate = 1 - delta.mean()

            if abs(rate - target_rate) < tol:
                return C

            if rate > target_rate:
                scale *= 1.2
            else:
                scale *= 0.8

        return C

    C = generate_censoring(Y, censoring_rate)

    T = np.minimum(Y, C)
    delta = (Y <= C).astype(int)

    l = T.copy()
    u = np.where(delta == 1, T, np.inf)

    df = pd.DataFrame({
        "X1": x1,
        "X2": x2,
        "X3": x3,
        "Y_true": Y,
        "C": C,
        "T_obs": T,
        "delta": delta,
        "l": l,
        "u": u
    })

    return df

# import numpy as np
# import pandas as pd

# from sksurv.svm import FastKernelSurvivalSVM
# from sksurv.metrics import concordance_index_censored
# from cvxopt import matrix, solvers

# def generate_wsvcr_data(n=500, censoring_rate=0.3, random_state=42):
#     np.random.seed(random_state)

#     x1 = np.random.normal(1, 1, n)
#     x2 = np.random.normal(2, np.sqrt(2), n)
#     x3 = np.random.exponential(1, n)

#     x2_safe = np.where(np.abs(x2) < 1e-8, 1e-8, x2)

#     lam = 5 * np.exp(
#         -x1 * np.log(np.abs(x2_safe)) + np.sin(x3 ** 2)
#     )

#     k = 3
#     U = np.random.uniform(0, 1, n)
#     Y = lam * (-np.log1p(-U)) ** (1 / k)

#     c_max = np.quantile(Y, 1 - censoring_rate)
#     C = np.random.uniform(0, c_max, n)

#     T = np.minimum(Y, C)
#     delta = (Y <= C).astype(int)

#     l = T.copy()
#     u = np.where(delta == 1, T, np.inf)

#     df = pd.DataFrame({
#         "X1": x1,
#         "X2": x2,
#         "X3": x3,
#         "Y_true": Y,
#         "T_obs": T,
#         "delta": delta,
#         "l": l,
#         "u": u
#     })

#     return df

In [3]:
solvers.options['show_progress'] = False

def polynomial_kernel(X, Y=None, degree=3, gamma=1.0, coef0=1.0):
    if Y is None:
        Y = X
    return (gamma * (X @ Y.T) + coef0) ** degree

def wavelet_kernel(X, Y=None, A=1.0):
    if Y is None:
        Y = X
    diff = X[:, None, :] - Y[None, :, :]
    return np.prod(
        np.cos(1.75 * diff / A) * np.exp(-0.5 * (diff / A) ** 2),
        axis=2
    )

def exponential_kernel(X, Y=None, gamma=1.0):
    if Y is None:
        Y = X
    diff = X[:, None, :] - Y[None, :, :]
    dist = np.sqrt(np.sum(diff**2, axis=2))
    return np.exp(-gamma * dist)

def cauchy_kernel(X, Y=None, gamma=1.0):
    if Y is None:
        Y = X
    diff = X[:, None, :] - Y[None, :, :]
    sq_dist = np.sum(diff**2, axis=2)
    return 1 / (1 + gamma * sq_dist)

def rbf_kernel(X, Y=None, gamma=1.0):
    if Y is None:
        Y = X
    diff = X[:, None, :] - Y[None, :, :]
    sq_dist = np.sum(diff**2, axis=2)
    return np.exp(-gamma * sq_dist)

def sigmoid_kernel(X, Y=None, gamma=1.0, coef0=0.0):
    if Y is None:
        Y = X
    return np.tanh(gamma * (X @ Y.T) + coef0)

def laplacian_kernel(X, Y=None, gamma=1.0):
    if Y is None:
        Y = X
    diff = np.abs(X[:, None, :] - Y[None, :, :])
    dist = np.sum(diff, axis=2)
    return np.exp(-gamma * dist)

def linear_kernel(X, Y=None):
    if Y is None:
        Y = X
    return X @ Y.T

class WSVCR_QP:
    def __init__(self, C=1.0, kernel="wavelet", kernel_params=None):
        self.C = C
        self.kernel = kernel
        self.kernel_params = kernel_params if kernel_params else {}

    def _compute_kernel(self, X, Y=None):

        if callable(self.kernel):
            return self.kernel(X, Y, **self.kernel_params)

        if self.kernel == "wavelet":
            return wavelet_kernel(X, Y, **self.kernel_params)

        elif self.kernel in "rbf":
            return rbf_kernel(X, Y, **self.kernel_params)

        elif self.kernel == "linear":
            return linear_kernel(X, Y)

        elif self.kernel == "poly":
            return polynomial_kernel(X, Y, **self.kernel_params)

        elif self.kernel == "sigmoid":
            return sigmoid_kernel(X, Y, **self.kernel_params)

        elif self.kernel == "laplacian":
            return laplacian_kernel(X, Y, **self.kernel_params)

        elif self.kernel == "exponential":
            return exponential_kernel(X, Y, **self.kernel_params)

        elif self.kernel == "cauchy":
            return cauchy_kernel(X, Y, **self.kernel_params)

        else:
            raise ValueError("Unknown kernel")

    def fit(self, X, l, u):
        X = np.asarray(X)
        l = np.asarray(l)
        u = np.asarray(u)

        n = X.shape[0]

        L = np.isfinite(l)
        U = np.isfinite(u)

        K = self._compute_kernel(X)

        P = np.block([
            [K, -K],
            [-K, K]
        ])

        P += 1e-8 * np.eye(2 * n)

        q = np.zeros(2 * n)

        for i in range(n):
            if L[i]:
                q[i] = -l[i]
            if U[i]:
                q[n + i] = u[i]

        A_eq = np.zeros((1, 2 * n))

        for i in range(n):
            if L[i]:
                A_eq[0, i] = 1
            if U[i]:
                A_eq[0, n + i] = -1

        b_eq = np.array([0.0])

        G = np.vstack([
            np.eye(2 * n),
            -np.eye(2 * n)
        ])

        h = np.hstack([
            self.C * np.ones(2 * n),
            np.zeros(2 * n)
        ])

        solution = solvers.qp(
            matrix(P),
            matrix(q),
            matrix(G),
            matrix(h),
            matrix(A_eq),
            matrix(b_eq)
        )

        z = np.array(solution['x']).flatten()

        self.alpha = z[:n]
        self.alpha_star = z[n:]
        self.theta = self.alpha - self.alpha_star

        self.X = X
        self.K = K

        self.b = self.compute_bias(l, u)

    def compute_bias(self, l, u):
        f = self.K @ self.theta

        b_vals = []

        for i in range(len(self.theta)):
            if self.alpha[i] > 1e-6 and np.isfinite(l[i]):
                b_vals.append(l[i] - f[i])
            elif self.alpha_star[i] > 1e-6 and np.isfinite(u[i]):
                b_vals.append(u[i] - f[i])

        return np.mean(b_vals) if b_vals else 0.0

    def predict(self, X_new):
        X_new = np.asarray(X_new)

        K_new = self._compute_kernel(self.X, X_new)

        return self.theta @ K_new + self.b

In [4]:
df = generate_wsvcr_data(n=300, censoring_rate=0.1, random_state=42)

X = df[["X1", "X2", "X3"]].values
l = df["l"].values
u = df["u"].values

model = WSVCR_QP(C=2.0, kernel="wavelet", kernel_params={"A": 1})
model.fit(X, l, u)

y_pred = model.predict(X)

In [5]:
def concordance_index(y_true, y_pred, delta):
    n = len(y_true)
    concordant = 0
    permissible = 0

    for i in range(n):
        for j in range(i + 1, n):

            if y_true[i] == y_true[j]:
                continue

            if delta[i] == 1 and y_true[i] < y_true[j]:
                permissible += 1
                if y_pred[i] < y_pred[j]:
                    concordant += 1
                elif y_pred[i] == y_pred[j]:
                    concordant += 0.5

            elif delta[j] == 1 and y_true[j] < y_true[i]:
                permissible += 1
                if y_pred[j] < y_pred[i]:
                    concordant += 1
                elif y_pred[i] == y_pred[j]:
                    concordant += 0.5

    return concordant / permissible if permissible > 0 else 0

In [ ]:
from sksurv.linear_model import CoxPHSurvivalAnalysis


def train_test_eval(df, C, A, kernel):
    X = df[["X1","X2","X3"]].values
    l = df["l"].values
    u = df["u"].values
    y = df["T_obs"].values
    delta = df["delta"].values

    X_train, X_test, l_train, l_test, u_train, u_test, y_train, y_test, d_train, d_test = train_test_split(
        X, l, u, y, delta, test_size=0.3, random_state=None, stratify=delta
    )

    model = WSVCR_QP(
        C=C,
        kernel=kernel,
        kernel_params={"A": A} if kernel == "wavelet" else {"gamma": A}
    )
    model.fit(X_train, l_train, u_train)

    y_pred = model.predict(X_test)

    cindex = concordance_index_censored(y_test, y_pred, d_test)
    # cindex = concordance_index(y_test, y_pred, d_test)

    return cindex

In [7]:
def grid_search(df):
    C_values = [0.25, 0.5, 1, 2, 4]
    A_values = [0.125, 0.25, 0.5, 1, 2, 4, 8]

    results = []

    for kernel in [
        "wavelet", "linear",
        "rbf", "exponential",
        "poly", "cauchy",
        "laplacian"]:
        for C in C_values:
            for A in A_values:
                scores = []

                for _ in range(100):
                    score = train_test_eval(
                        df,
                        C,
                        A,
                        kernel)
                    scores.append(score)

                results.append({
                    "kernel": kernel,
                    "C": C,
                    "A": A if kernel == "wavelet" else None,
                    "gamma": A if kernel != "wavelet" else None,
                    "mean_cindex": np.mean(scores),
                    "std": np.std(scores)
                })

                print(kernel, C, A, np.mean(scores))

    return pd.DataFrame(results)

In [8]:
# results = []

# for censor in [0.06, 0.25, 0.5]:
#     df = generate_wsvcr_data(n=30, censoring_rate=censor)
#     results_grid = grid_search(df).assign(prop_censor=censor)
#     results.append(results_grid)

In [8]:
# pd.concat(results)\
#     .query("prop_censor != 0.95")\
#     .sort_values(
#         "mean_cindex",
#         ascending=False
#     ).head(40)

### Real Data Application

In [ ]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

from sksurv.datasets import (
    load_veterans_lung_cancer,
    load_whas500,
    load_gbsg2,
    load_flchain
)

from sksurv.metrics import concordance_index_censored


def load_survival_dataset(loader):
    X_raw, y_struct = loader()

    X = pd.DataFrame(X_raw)

    X = pd.get_dummies(X, drop_first=True)

    event_col, time_col = y_struct.dtype.names

    df = X.copy()
    df["event"] = y_struct[event_col].astype(int)
    df["time"] = y_struct[time_col]

    return df


def to_interval_format(df):
    T = df["time"].values
    delta = df["event"].values

    l = T.copy()
    u = np.where(delta == 1, T, np.inf)

    X = df.drop(columns=["time", "event"]).values

    return X, l, u


def train_test_eval(df, C, A, kernel):

    X, l, u = to_interval_format(df)

    X_train, X_test, l_train, l_test, u_train, u_test = train_test_split(
        X, l, u,
        test_size=0.3
    )

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    params = {"A": A} if kernel == "wavelet" else {"gamma": A}

    model = WSVCR_QP(
        C=C,
        kernel=kernel,
        kernel_params=params
    )

    model.fit(X_train, l_train, u_train)

    y_pred = model.predict(X_test)

    risk = -y_pred

    event_test = np.isfinite(u_test)
    time_test = l_test

    cindex = concordance_index_censored(
        event_test,
        time_test,
        risk
    )[0]

    return cindex


def grid_search_real(df, n_repeats=10):

    C_values = [0.25, 0.5, 1, 2, 4]
    A_values = [0.125, 0.25, 0.5, 1, 2, 4, 8]

    kernels = [
        "wavelet",
        "linear",
        "rbf",
        "exponential",
        "poly",
        "cauchy",
        "laplacian"
    ]

    results = []

    for kernel in kernels:
        for C in C_values:
            for A in A_values:

                scores = []

                for _ in range(n_repeats):
                    try:
                        score = train_test_eval(df, C, A, kernel)
                        scores.append(score)
                    except Exception as e:
                        print(f"{kernel} failed: {e}")

                if len(scores) > 0:
                    mean_score = np.mean(scores)
                    std_score = np.std(scores)

                    print(kernel, C, A, mean_score)

                    results.append({
                        "kernel": kernel,
                        "C": C,
                        "A": A if kernel == "wavelet" else None,
                        "gamma": A if kernel != "wavelet" else None,
                        "mean_cindex": mean_score,
                        "std": std_score
                    })

    return pd.DataFrame(results)


datasets = {
    "veterans": load_veterans_lung_cancer,
    "whas500": load_whas500,
    "gbsg2": load_gbsg2,
    # "flchain": load_flchain
}

all_results = []

for dataset_name, loader in datasets.items():
    print(f"\n{'='*50}")
    print(f"DATASET: {dataset_name}")
    print(f"{'='*50}")

    df = load_survival_dataset(loader)

    results = grid_search_real(df, n_repeats=10)
    results["dataset"] = dataset_name

    all_results.append(results)


final_results = pd.concat(all_results, ignore_index=True)

print("\nTOP RESULTS OVERALL:")
print(
    final_results
    .sort_values("mean_cindex", ascending=False)
    .head(20)
)

print("\nBEST PER DATASET:")
print(
    final_results
    .sort_values("mean_cindex", ascending=False)
    .groupby("dataset")
    .head(5)
)



DATASET: veterans
wavelet 0.25 0.125 0.5447025563299637
wavelet 0.25 0.25 0.5158495046205618
wavelet 0.25 0.5 0.4656305688912791
wavelet 0.25 1 0.5247117171298139
wavelet 0.25 2 0.584876590601918
wavelet 0.25 4 0.6817170092416439
wavelet 0.25 8 0.7087078021799856
wavelet 0.5 0.125 0.5178388950988089
wavelet 0.5 0.25 0.5042327006692152
wavelet 0.5 0.5 0.474878961352148
wavelet 0.5 1 0.5346813481322128
wavelet 0.5 2 0.6059431241852968
wavelet 0.5 4 0.6800314567152352
wavelet 0.5 8 0.7119945811617672
wavelet 1 0.125 0.5243400664031941
wavelet 1 0.25 0.474783368161565
wavelet 1 0.5 0.481281171143179
wavelet 1 1 0.5643315812814282
wavelet 1 2 0.5817479052113304
wavelet 1 4 0.6618994943118639
wavelet 1 8 0.7160175302686909
wavelet 2 0.125 0.5117699658585971
wavelet 2 0.25 0.5031377836959267
wavelet 2 0.5 0.5010967831963098
wavelet 2 1 0.5592174829601142
wavelet 2 2 0.5897197966341448
wavelet 2 4 0.6716709612677515
wavelet 2 8 0.6784061610790975
wavelet 4 0.125 0.519662039711146
wavelet 4 0.